In [1]:
import os

In [2]:
os.chdir("C:/Users/Shubham Joshi/Desktop/nlp project")

In [3]:
from dataclasses import dataclass
from pathlib import Path
import os
@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: Path



In [4]:
from textSummarizer.utils.common import read_yaml,create_dir
from textSummarizer.constants import CONFIG_FILE_PATH, PARAMS_FILE_PATH
from datasets import load_dataset,load_from_disk
from transformers import AutoTokenizer
import os
from textSummarizer.logging import logger
from textSummarizer.exception.__init__ import CustomException
import sys

c:\Users\Shubham Joshi\Desktop\nlp project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
class ConfigurationManager:
    def __init__(self,
                config_file_path=CONFIG_FILE_PATH,
                params_file_path=PARAMS_FILE_PATH):
        self.config=read_yaml(config_file_path)
        self.params=read_yaml(params_file_path)

        create_dir([self.config.artifacts_root])

    def get_data_transformation_config(self)->DataTransformationConfig:
        config=self.config.data_transformation
        create_dir([config.root_dir])
        data_transformation_config=DataTransformationConfig(
            root_dir=config.root_dir,
            data_path=config.data_path, 
            tokenizer_name=config.tokenizer_name
        )
        return data_transformation_config


In [12]:
class DataTransformation:
    def __init__(self,config:DataTransformationConfig):

        self.config=config
        self.tokenizer=AutoTokenizer.from_pretrained(self.config.tokenizer_name)
        
    
    def convert_examples_to_features(self, example_batch):
    
        input_encodings = self.tokenizer(example_batch['dialogue'], max_length=1024, truncation=True)
        
        
        target_encodings = self.tokenizer(text_target=example_batch['summary'], max_length=128, truncation=True)
        
        return {
            'input_ids': input_encodings['input_ids'],
            'attention_mask': input_encodings['attention_mask'],
            'labels': target_encodings['input_ids']
        }
    
    def convert(self):
        dataset_samsum=load_from_disk(dataset_path=self.config.data_path)
        dataset_samsum_pt=dataset_samsum.map(self.convert_examples_to_features,batched=True)
        save_path = os.path.join(self.config.root_dir, "samsum_dataset")
        dataset_samsum_pt.save_to_disk(save_path)

In [13]:
try:
    config=ConfigurationManager()
    data_transformation_config=config.get_data_transformation_config()
    data_transformation=DataTransformation(config=data_transformation_config)
    data_transformation.convert()
except Exception as e:
    raise CustomException(e,sys)

[2026-05-11 20:31:13,585:INFO: common: yaml file read successfully]
[2026-05-11 20:31:13,587:INFO: common: yaml file read successfully]
[2026-05-11 20:31:13,589:INFO: common: Directory created successfully at ['artifacts']]
[2026-05-11 20:31:13,590:INFO: common: Directory created successfully at ['artifacts/data_transformation']]
[2026-05-11 20:31:13,904:INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 20:31:13,920:INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-05-11 20:31:14,204:INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-05-11 20:31:14,222:INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/g

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<00:00, 120269.94 examples/s]
